# Serveur Ollama sur le GPU de Colab (pour le labo)

**Kabyle LLM Benchmark, phase 2.** Quand le PC n'a pas assez de mémoire pour faire tourner Qwen, ce notebook le fait tourner sur le **GPU gratuit de Colab**. Le labo reste sur le PC : il prépare les questions (few-shot, RAG, contrôle anti-contamination) et envoie seulement la question finale à ce serveur, par un tunnel chiffré (Cloudflare).

## Ce qui part sur Colab

- **Uniquement les tests publics**, déjà publiés sur GitHub. Dès que l'adresse d'Ollama n'est plus celle du PC, le labo marque tous les modèles Ollama « tests publics seulement » et refuse tout run qui leur enverrait un test privé.
- Les exemples few-shot et RAG, tirés des corpus publics décontaminés.

## Étapes

1. *Exécution → Modifier le type d'exécution → T4 GPU*.
2. **Modèle fine-tuné (facultatif)** : à la fin du notebook `01_qlora_finetune`, copier le `.gguf` dans Google Drive (voir `notebooks/README.md`). Laisser `GGUF_ON_DRIVE` vide sinon.
3. *Exécution → Tout exécuter*. La dernière cellule affiche une adresse `https://….trycloudflare.com`.
4. Sur le PC, dans `.env` : `OLLAMA_DOCKER_URL=<cette adresse>`, puis `docker compose up -d api`.
5. **Laisser cet onglet ouvert** pendant tout le run. Le serveur s'arrête avec la session Colab.

L'adresse du tunnel est aléatoire mais publique : ne la partager avec personne. Elle disparaît à la fin de la session.


In [ ]:
# ── Configuration ──────────────────────────────────────────
MODELS = ["qwen3.5:4b", "bge-m3"]   # mêmes modèles que le labo
GGUF_ON_DRIVE = ""                   # ex. "kab-llm/kab-qwen3.5-4b.Q4_K_M.gguf" (chemin dans Mon Drive)
FINETUNED_NAME = "kab-qwen3.5-4b"    # nom attendu par models.yaml (qwen3.5-4b-lora)

In [ ]:
# ── Installer et démarrer Ollama ───────────────────────────
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import os, subprocess, time, requests
env = dict(os.environ, OLLAMA_KEEP_ALIVE="24h", OLLAMA_MAX_LOADED_MODELS="2")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        print("Ollama", requests.get("http://127.0.0.1:11434/api/version", timeout=2).json()); break
    except Exception:
        time.sleep(1)

In [ ]:
# ── Télécharger les modèles ────────────────────────────────
for name in MODELS:
    !ollama pull {name}
for m in requests.get("http://127.0.0.1:11434/api/tags").json()["models"]:
    print(m["name"], m["digest"][:12])  # à comparer avec « ollama list » sur le PC

In [ ]:
# ── Modèle fine-tuné depuis Google Drive (facultatif) ─────
if GGUF_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    gguf = f"/content/drive/MyDrive/{GGUF_ON_DRIVE}"
    assert os.path.exists(gguf), f"introuvable : {gguf}"
    # Même gabarit de conversation que le modèle de base : on reprend son Modelfile
    base = subprocess.run(["ollama", "show", MODELS[0], "--modelfile"], capture_output=True, text=True).stdout
    lines = [l for l in base.splitlines() if not l.startswith("FROM ")]
    open("Modelfile", "w").write(f"FROM {gguf}\n" + "\n".join(lines) + "\n")
    !ollama create {FINETUNED_NAME} -f Modelfile
    !ollama list
else:
    print("Pas de modèle fine-tuné : seulement", MODELS)

In [ ]:
# ── Préchauffage sur le GPU (vérifie que tout répond) ──────
r = requests.post("http://127.0.0.1:11434/api/chat", json={
    "model": MODELS[0], "messages": [{"role": "user", "content": "Bonjour"}],
    "stream": False, "think": False}, timeout=600)
print(r.json()["message"]["content"][:200])
!ollama ps

In [ ]:
# ── Tunnel Cloudflare : adresse à mettre dans .env ─────────
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import re
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:11434",
     "--http-host-header", "localhost:11434"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0); break
print("\n=== Dans .env sur le PC ===")
print(f"OLLAMA_DOCKER_URL={URL}")
print("puis : docker compose up -d api")

In [ ]:
# ── Garder la session active et suivre l'activité ──────────
# Laisser tourner pendant le run (arrêter avec le bouton ■ à la fin)
import datetime
while True:
    ps = requests.get("http://127.0.0.1:11434/api/ps").json().get("models", [])
    print(datetime.datetime.now().strftime("%H:%M"), "modèles chargés :", [m["name"] for m in ps])
    time.sleep(300)